# ncpu-computer

Train one shared local NCA rule on a ternary string task, then measure full-tape and interpreted correctness across time and tape lengths. The target is used only by the loss.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'run':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from ncpu_computer import (
    ExperimentConfig, TaskDataset,
    addition_task, evaluate, load_model, train_seeds, validate_experiment,
)
from ncpu_computer.evaluation import format_results

In [ ]:
config = ExperimentConfig()
train_task = addition_task(4)
train_data = TaskDataset.from_task(train_task, config.geometry.tape_slots)

print(validate_experiment(config, train_data))
print(f'input example:  {train_data.input_strings[-1]}')
print(f'target example: {train_data.target_strings[-1]}')

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    seed_results = train_seeds(
        config, train_data, seeds=(0,), checkpoint_dir=ROOT / 'checkpoints'
    )
    seed_results

In [ ]:
checkpoint_path = ROOT / 'checkpoints' / 'best.pt'

if checkpoint_path.is_file():
    model, trained_config, checkpoint = load_model(checkpoint_path)
    result_4bit = evaluate(
        model, trained_config.geometry, train_data,
        steps=trained_config.training.rollout_steps,
        step_start=trained_config.training.supervision_start,
        step_end=trained_config.training.supervision_end,
        batch_size=trained_config.training.batch_size,
    )
    print(format_results([result_4bit]))
else:
    print(f'No checkpoint yet: {checkpoint_path}')

In [ ]:
RUN_8BIT_EVALUATION = False

if RUN_8BIT_EVALUATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path)
    geometry_8bit = replace(trained_config.geometry, tape_slots=20)
    data_8bit = TaskDataset.from_task(addition_task(8), geometry_8bit.tape_slots)
    result_8bit = evaluate(
        model, geometry_8bit, data_8bit,
        steps=trained_config.training.rollout_steps,
        step_start=trained_config.training.supervision_start,
        step_end=trained_config.training.supervision_end,
        batch_size=trained_config.training.batch_size,
    )
    print(format_results([result_8bit]))